# NeRF

In [ ]:
!rm -rf '/content/cg_nerf/'
!git clone https://github.com/LazyGH/cg_nerf.git
%cd /content/cg_nerf
# %pip install -U imageio imageio-ffmpeg scipy matplotlib lpips configargparse tensorboard

# !sed -i 's/archive.ubuntu.com/us-central1.gce.archive.ubuntu.com/g' /etc/apt/sources.list
# !apt-get update -y
# !apt-get install -y imagemagick

%pip install -q condacolab
import condacolab
condacolab.install()

%cd /content/cg_nerf
!conda env create -f nerf/environment_colab_tf115.yml
!conda run -n nerf-tf115 python -V

%cd /content/cg_nerf/nerf
!conda run -n nerf-tf115 bash download_example_data.sh

Train test

In [ ]:
%cd /content/cg_nerf/nerf
!time conda run -n nerf-tf115 python run_nerf.py --config course_configs/lego_t4_test.txt

Train NeRF Lego

In [ ]:
%cd /content/cg_nerf/nerf
!time conda run -n nerf-tf115 python run_nerf.py --config course_configs/lego_t4_20min.txt

Train NeRF Fern

In [ ]:
%cd /content/cg_nerf/nerf
!time conda run -n nerf-tf115 python run_nerf.py --config course_configs/fern_t4_20min.txt

Tensorboard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/cg_nerf/nerf/logs/summaries

NeRF Evaluation

Trained Lego

In [ ]:
%cd /content/cg_nerf
!conda run -n nerf-tf115 python project_tools/render_eval_nerf.py --config nerf/course_configs/lego_t4_20min.txt --output_dir result/nerf/lego_trained --source trained --train_time_minutes 20

Trained Fern

In [ ]:
%cd /content/cg_nerf
!conda run -n nerf-tf115 python project_tools/render_eval_nerf.py --config nerf/course_configs/fern_t4_20min.txt --output_dir result/nerf/fern_trained --source trained --train_time_minutes 20

download paper pre-trained model

In [ ]:
%cd /content/cg_nerf/nerf
!conda run -n nerf-tf115 bash download_example_weights.sh

Paper Lego

In [ ]:
%cd /content/cg_nerf
!conda run -n nerf-tf115 python project_tools/render_eval_nerf.py --config nerf/course_configs/lego_t4_20min.txt --ckpt nerf/logs/lego_example/model_fine_200000.npy --output_dir result/nerf/lego_pretrained --source pretrained

Paper Fern

In [ ]:
%cd /content/cg_nerf
!conda run -n nerf-tf115 python project_tools/render_eval_nerf.py --config nerf/course_configs/fern_t4_30min.txt --ckpt nerf/logs/lego_example/model_fine_200000.npy --output_dir result/nerf/fern_pretrained  --source pretrained

Savedata

In [ ]:
from google.colab import drive
drive.mount('/content/drive/My Drive/Colab Notebooks/CG-pj')
!cp -r /content/cg_nerf/nerf/logs/* /content/drive/My Drive/Colab Notebooks/CG-pj/nerf/logs/
!cp -r /content/cg_nerf/DirectVoxGO/logs/* /content/drive/My Drive/Colab Notebooks/CG-pj/DirectVoxGO/logs/
!cp -r /content/cg_nerf/result /content/drive/My Drive/Colab Notebooks/CG-pj/result/

# DVGO

In [ ]:
%cd /content/cg_nerf
!pip install -U torch torchvision ninja tqdm opencv-python-headless imageio imageio-ffmpeg scipy lpips torch_efficient_distloss einops openmim mmcv-lite

import torch
import os
os.environ['TORCH'] = torch.__version__.split('+')[0]
os.environ['CUDA'] = 'cpu' if torch.version.cuda is None else f"cu{torch.version.cuda.replace('.', '')}"

# 4. Verify they were set
print("TORCH:", os.environ['TORCH'])
print("CUDA:", os.environ['CUDA'])

!pip install torch_scatter -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html

%cd /content/cg_nerf
!mkdir -p DirectVoxGO/data
!cp -r nerf/data/nerf_synthetic DirectVoxGO/data/
!cp -r nerf/data/nerf_llff_data DirectVoxGO/data/

%cd /content/cg_nerf/DirectVoxGO
from lib import dvgo
print('DVGO CUDA extension loaded successfully.')

Train Test

In [ ]:
%cd /content/cg_nerf/DirectVoxGO
!time python run.py --config configs/course/fern_t4_test.py

Train DVGO lego

In [ ]:
%cd /content/cg_nerf/DirectVoxGO
!time python run.py --config configs/course/lego_t4_20min.py --i_tb 100

Train DVGO fern

In [ ]:
%cd /content/cg_nerf/DirectVoxGO
!time python run.py --config configs/course/fern_t4_20min.py --i_tb 100

Tensorboard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/cg_nerf/DirectVoxGO/logs

DVGO Evaluation

Lego

In [ ]:
%cd /content/cg_nerf
!python project_tools/render_eval_dvgo.py --config DirectVoxGO/configs/course/lego_t4_20min.py --output_dir result/dvgo/lego_trained --source traine--train_time_minutes 20

Fern

In [ ]:
%cd /content/cg_nerf
!python project_tools/render_eval_dvgo.py --config DirectVoxGO/configs/course/fern_t4_20min.py --output_dir result/dvgo/fern_trained --source trained --train_time_minutes 20

Summary quantitative Results

In [ ]:
%%bash
cd /content/cg_nerf
python project_tools/aggregate_metrics.py \
  result/nerf/lego_trained/metrics.json \
  result/nerf/fern_trained/metrics.json \
  result/nerf/lego_pretrained/metrics.json \
  result/nerf/fern_pretrained/metrics.json \
  result/dvgo/lego_trained/metrics.json \
  result/dvgo/fern_trained/metrics.json \
  --out_csv result/summary_metrics.csv

Save data

In [ ]:
from google.colab import drive
drive.mount('/content/drive/My Drive/Colab Notebooks/CG-pj')
!cp -r /content/cg_nerf/nerf/logs/* /content/drive/My Drive/Colab Notebooks/CG-pj/nerf/logs/
!cp -r /content/cg_nerf/DirectVoxGO/logs/* /content/drive/My Drive/Colab Notebooks/CG-pj/DirectVoxGO/logs/
!cp -r /content/cg_nerf/result /content/drive/My Drive/Colab Notebooks/CG-pj/result/